In [ ]:
import pandas as pd

from preprocessing.geneva_stroke_unit_preprocessing.imaging_preprocessing.imaging_data_preprocessing import preprocess_imaging_data
from preprocessing.geneva_stroke_unit_preprocessing.stroke_registry_params_preprocessing.utils import set_sample_date
from preprocessing.geneva_stroke_unit_preprocessing.utils import create_registry_case_identification_column
from preprocessing.geneva_stroke_unit_preprocessing.patient_selection.restrict_to_patient_selection import restrict_to_patient_selection


In [ ]:
imaging_data_path = '/Users/jk1/temp/opsum_end/imaging_extraction/combined_perfusion_parameters_20260122_180009.csv'
patient_selection_path = '/Users/jk1/temp/opsum_end/gsu_extraction_09052025_204357/high_frequency_data_patient_selection_with_details.csv'
stroke_registry_data_path = '/Users/jk1/stroke_datasets/stroke_registry_post_hoc_modified.xlsx'

In [ ]:
# Load stroke registry data and restrict to patient selection
stroke_registry_df = pd.read_excel(stroke_registry_data_path)
stroke_registry_df['patient_id'] = stroke_registry_df['Case ID'].apply(lambda x: x[8:-4])
stroke_registry_df['EDS_last_4_digits'] = stroke_registry_df['Case ID'].apply(lambda x: x[-4:])
stroke_registry_df['case_admission_id'] = create_registry_case_identification_column(stroke_registry_df)

# set sample date to stroke onset or arrival at hospital, whichever is later
stroke_registry_df = set_sample_date(stroke_registry_df)

restricted_stroke_registry_df = restrict_to_patient_selection(stroke_registry_df, patient_selection_path,
                                                                verbose=True, restrict_to_event_period=False)


In [ ]:
imaging_data_df = pd.read_csv(imaging_data_path)

In [ ]:
imaging_data_df.head()

In [ ]:
preprocessed_imaging_data_df = preprocess_imaging_data(imaging_data_path, patient_selection_path, restricted_stroke_registry_df)

In [ ]:
preprocessed_imaging_data_df.head()

In [ ]:
# convert to wide format: one row per case_admission_id & sample_date, one column per sample_label
imaging_wide_df = (
    preprocessed_imaging_data_df
    .pivot_table(
        index=['case_admission_id', 'sample_date'],
        columns='sample_label',
        values='value',
        aggfunc='first'
    )
    .reset_index()
    .rename_axis(None, axis=1)
    .sort_values(['case_admission_id', 'sample_date'])
)
imaging_wide_df.head()